# 00 - Inventario de fuentes

Auditoría de los 5 archivos de `data/raw/`. Este notebook solo audita:
la limpieza real vive en `src/ingest.py` (`load_base_accidentes`) y
`src/clean.py`. Ver el detalle completo en `reports/diccionario_datos.md`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import RAW_DIR

for f in sorted(RAW_DIR.glob("*.xlsx")):
    print(f.name)
    print("  hojas:", pd.ExcelFile(f).sheet_names)

## Hallazgo: solo 2 de los 5 archivos tienen registros fila-por-accidente

- `Base de Accidentes 2023 _ Total.xlsx` (hoja `Base`)
- `Base Accidentes 2024_Todo Alicorp.xlsx` (hoja `Base`)

Los otros 3 (`BASE CALCULO INDICADORES 2024.xlsx`, `Resultados SST 2023
v06final.xlsx`, `Tablero de Accidentes e incidentes.xlsx`) son tableros/
indicadores agregados, no auditados a fondo todavía — ver la tabla de
fuentes en `reports/diccionario_datos.md` para el detalle y lo pendiente.

In [ ]:
from src.ingest import load_raw

df23_raw = load_raw("Base de Accidentes 2023 _ Total.xlsx", sheet_name="Base")
df24_raw = load_raw("Base Accidentes 2024_Todo Alicorp.xlsx", sheet_name="Base")

print("2023 crudo:", df23_raw.shape, "-> filas con Fecha real:", df23_raw["Fecha"].notna().sum())
print("2024 crudo:", df24_raw.shape)

## Hallazgos de calidad de datos (documentados en `reports/diccionario_datos.md`)

1. **2023**: de 306 filas, solo 107 son datos reales. Las 199 restantes son
   residuo de formato de Excel (solo la columna `Sem` traía valor).
2. **2024**: 11 columnas `Unnamed: 46..56` vacías (artefacto de formato),
   descartadas.
3. **Fechas mixtas**: algunas celdas de `Fecha` no tenían formato de fecha
   en el Excel original y llegaban como número de serie de Excel en vez de
   `datetime` — `src/ingest.py` lo corrige (`_parse_mixed_excel_date`).
4. **Nombres/apellidos**: se reemplazan por `id_persona` (hash) antes de
   guardar nada — nunca se escribe un nombre real fuera de `data/raw/`.

In [ ]:
from src.clean import report_nulls
from src.ingest import load_base_accidentes, save_processed

df23 = load_base_accidentes("2023")
df24 = load_base_accidentes("2024")

print("2023 limpio:", df23.shape)
print("2024 limpio:", df24.shape)

save_processed(df23, "accidentes_2023")
save_processed(df24, "accidentes_2024")

In [ ]:
# Nulos por columna (NO se imputa nada aqui, solo se reporta).
# Ver decision de cada columna en reports/diccionario_datos.md.
print("--- 2023 ---")
display(report_nulls(df23))
print("--- 2024 ---")
display(report_nulls(df24))

## Pendiente

- [ ] Auditar los 3 archivos restantes (indicadores/tableros) y decidir si
      aportan algo que no esté ya en las 2 bases de accidentes.
- [ ] Decidir con el equipo qué hacer con las columnas de `accidentes_2023`
      que superan 80% de nulos (ver `reports/diccionario_datos.md`).
- [ ] Revisar `descripcion_accidente` antes de compartirla (puede mencionar
      nombres en el texto libre — no se anonimizó automáticamente).